In [ ]:
import numpy as np
from pathlib import Path
import pandas as pd
import yaml

def extract_runs(path: Path):
    with open(path) as f:
        lines = [l.strip() for l in f if l.strip()]

    runs = []
    i = 0

    while i < len(lines):
        if lines[i].startswith("MEAN"):
            mean_rows = [lines[i + j].split(",") for j in range(1, 5)]

            # find STD block right after
            if lines[i + 5].startswith("STD"):
                std_rows = [lines[i + 5 + j].split(",") for j in range(1, 5)]
                runs.append((mean_rows, std_rows))
                i += 10
            else:
                raise ValueError(f"STD block missing after MEAN in {path}")
        else:
            i += 1

    return runs


def compute_metrics(paths: list[Path]):
    metrics = {
        "FALSE": {"f1": [], "f1_std": []},
        "TRUE": {"f1": [], "f1_std": []},
        "MACRO": {"f1": [], "f1_std": []},
        "WEIGHTED": {
            "f1": [], "f1_std": [],
            "acc": [], "acc_std": [],
            "auc": [], "auc_std": [],
            "ap": [], "ap_std": []
        },
    }

    for path in paths:
        runs = extract_runs(path)

        for mean_rows, std_rows in runs:
            for mean_r, std_r in zip(mean_rows, std_rows):
                cls = mean_r[0]

                precision, recall, f1, accuracy, auc_val, ap_val = map(float, mean_r[1:])
                _, _, f1_std, acc_std, auc_std, ap_std = map(float, std_r[1:])

                metrics[cls]["f1"].append(f1)
                metrics[cls]["f1_std"].append(f1_std)

                if cls == "WEIGHTED":
                    metrics[cls]["acc"].append(accuracy)
                    metrics[cls]["acc_std"].append(acc_std)
                    metrics[cls]["auc"].append(auc_val)
                    metrics[cls]["auc_std"].append(auc_std)
                    metrics[cls]["ap"].append(ap_val)
                    metrics[cls]["ap_std"].append(ap_std)

    def fmt(mean_vals, std_vals):
        if not mean_vals:
            return "nan ± nan"
        return f"{np.mean(mean_vals):.2f} ± {np.mean(std_vals):.2f}"

    return [
        fmt(metrics["FALSE"]["f1"], metrics["FALSE"]["f1_std"]),
        fmt(metrics["TRUE"]["f1"], metrics["TRUE"]["f1_std"]),
        fmt(metrics["MACRO"]["f1"], metrics["MACRO"]["f1_std"]),
        fmt(metrics["WEIGHTED"]["f1"], metrics["WEIGHTED"]["f1_std"]),
        fmt(metrics["WEIGHTED"]["acc"], metrics["WEIGHTED"]["acc_std"]),
        fmt(metrics["WEIGHTED"]["auc"], metrics["WEIGHTED"]["auc_std"]),
        fmt(metrics["WEIGHTED"]["ap"], metrics["WEIGHTED"]["ap_std"]),
    ]


base = Path("results/meds")
rows = []

with open("experiments.yaml", "r") as f:
    exp_config = yaml.safe_load(f)["experiments"]

    for group_name, experiments in exp_config.items():
        print(f"\n=== Group: {group_name} ===")

        for exp in experiments:
            print(f"Running task: {exp['task']}")

            if exp['task'] != "first_24_in_hospital_mortality":
                continue

            ifiles = [
                Path(f"{base}/{exp['task']}/{i}/metrics_TS_{exp['sample_size']}.csv")
                for i in range(exp['num_of_samples'])
            ]

            values = compute_metrics(ifiles)

            row = {
                "task": exp["task"],
                "F1_false": values[0],
                "F1_true": values[1],
                "F1_macro": values[2],
                "F1_weighted": values[3],
                "Accuracy": values[4],
                "AUC": values[5],
                "AP": values[6],
            }

            rows.append(row)

pd.DataFrame(rows).to_csv("summary_metrics.tsv", index=False, sep="\t")